# JASSIM AI — Google Colab GPU Server
دفتر تشغيل تجريبي لخادم JASSIM AI. يحفظ SQLite في Google Drive، ويشغّل Ollama والنموذج، ثم يفتح نفق HTTPS مؤقتاً عبر Cloudflare.

**ملاحظات:** جلسات Colab المجانية مؤقتة وGPU غير مضمون. لا تشارك رابط النفق أو رمز API مع الآخرين. لا تضع أسراراً في GitHub.

من قائمة Runtime اختر Change runtime type ثم GPU إن كان متاحاً.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, time, json, secrets, getpass, requests, re, textwrap

PROJECT = Path('/content/JASSIM-AI-Android')
DATA_DIR = Path('/content/drive/MyDrive/JASSIM_AI/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Drive mounted. Persistent database directory:', DATA_DIR)


## 1. تنزيل ملفات خادم JASSIM
نستخدم نسخة المستودع الحالية دون تعديل ملفات التطبيق أو الذاكرة الموجودة فيه.

In [ ]:
if not PROJECT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/stevennpllk-crypto/-JASSIM-AI-Android.git',str(PROJECT)], check=True)
SERVER = PROJECT / 'backend' / 'server.py'
assert SERVER.exists(), 'backend/server.py is missing'
print('Server source ready:', SERVER)


## 2. إعداد رمز الحماية
أنشئ رمزاً عشوائياً أو أدخل رمزاً تملكه. سيُستخدم في التطبيق والموقع. لا ترسله في المحادثة ولا تحفظه في GitHub.

In [ ]:
API_TOKEN = secrets.token_urlsafe(48)
print('Generated a private API token for this Colab session.')
print('Keep this notebook session private. Copy the token from the next cell output only if you need to configure the Android app/dashboard.')
print(API_TOKEN)

os.environ['JASSIM_API_TOKEN'] = API_TOKEN
os.environ['JASSIM_CORS_ORIGIN'] = 'https://stevennpllk-crypto.github.io'
os.environ['JASSIM_HOST'] = '127.0.0.1'
os.environ['JASSIM_PORT'] = '8787'
os.environ['JASSIM_DATA_DIR'] = str(DATA_DIR)
os.environ['JASSIM_OLLAMA_URL'] = 'http://127.0.0.1:11434'
os.environ['JASSIM_OLLAMA_MODEL'] = 'qwen2.5:1.5b'


## 3. تثبيت وتشغيل Ollama
نستخدم نموذجاً صغيراً نسبياً كبداية. تنزيل النموذج يحتاج اتصال إنترنت ووقتاً ومساحة تخزين.

In [ ]:
install = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, text=True, capture_output=True)
print(install.stdout[-3000:])
if install.returncode != 0:
    print(install.stderr[-3000:])
    raise RuntimeError('Ollama installation failed')

ollama_log = open('/content/ollama.log','w')
ollama_proc = subprocess.Popen(['ollama','serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get('http://127.0.0.1:11434/api/tags', timeout=2)
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not start. Check /content/ollama.log')

pull = subprocess.run(['ollama','pull','qwen2.5:1.5b'], text=True)
if pull.returncode != 0:
    raise RuntimeError('Model download failed')
print('Ollama and qwen2.5:1.5b are ready.')


## 4. تشغيل JASSIM API مع ذاكرة Drive
تُحفظ قاعدة البيانات داخل مجلد JASSIM_AI في Drive. لا تحذف هذا المجلد إذا أردت الاحتفاظ بالمحادثات.

In [ ]:
api_log = open('/content/jassim-api.log','w')
api_proc = subprocess.Popen(['python',str(SERVER)], cwd=str(PROJECT/'backend'), env=os.environ.copy(), stdout=api_log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        health = requests.get('http://127.0.0.1:8787/health', timeout=2).json()
        if health.get('ok'): break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('JASSIM API did not start. Check /content/jassim-api.log')
print(json.dumps(health, ensure_ascii=False, indent=2))


## 5. اختبار المحرك والحماية
هذا الاختبار يرسل رسالة تجريبية إلى النموذج ويحفظها في سجل المحادثة.

In [ ]:
headers = {'Authorization': 'Bearer ' + API_TOKEN, 'Content-Type':'application/json'}
test = requests.post('http://127.0.0.1:8787/api/chat', headers=headers, json={'message':'عرّف بنفسك في جملة قصيرة بالعربية.','conversation_id':'colab-check'}, timeout=180)
print('HTTP:', test.status_code)
print(json.dumps(test.json(), ensure_ascii=False, indent=2))
assert test.ok, 'API chat test failed'


## 6. فتح رابط HTTPS مؤقت
سنشغّل Cloudflare Quick Tunnel. الرابط يتغير عند إعادة تشغيل الجلسة، وهو ليس استضافة دائمة.

In [ ]:
deb = '/content/cloudflared.deb'
subprocess.run(['bash','-lc',f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O {deb} && dpkg -i {deb}"], check=True)
cf_log = open('/content/cloudflared.log','w')
cf_proc = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url','http://127.0.0.1:8787'], stdout=cf_log, stderr=subprocess.STDOUT, text=True)
tunnel_url = None
for _ in range(60):
    time.sleep(2)
    try:
        log_text = Path('/content/cloudflared.log').read_text(errors='ignore')
        match = re.search(r'https://[a-z0-9-]+\\.trycloudflare\\.com', log_text)
        if match:
            tunnel_url = match.group(0)
            break
    except Exception:
        pass
if not tunnel_url:
    print(Path('/content/cloudflared.log').read_text(errors='ignore')[-4000:])
    raise RuntimeError('Could not obtain a Cloudflare tunnel URL')
print('JASSIM API HTTPS URL:', tunnel_url)
print('Health:', requests.get(tunnel_url + '/health', timeout=20).json())


## 7. ربط التطبيق والموقع
في إعدادات تطبيق Android ولوحة التحكم:

- Server URL: انسخ رابط HTTPS الذي ظهر في الخلية السابقة.
- API Token: انسخ الرمز الخاص الذي وُلّد في خلية الحماية.

احتفظ بالرابط والرمز بشكل خاص. لا تنشرهما ولا ترسلهما إلى أشخاص آخرين. عند انتهاء جلسة Colab أعد تشغيل الخلايا؛ قد تحصل على رابط جديد، لكن بيانات SQLite تبقى في Drive.

**مهم:** لا تترك الخلية الأخيرة دون مراقبة؛ إيقاف جلسة Colab يوقف النموذج والخادم والنفق.

In [ ]:
print('API URL:', tunnel_url)
print('API Token:', API_TOKEN)
print('Persistent DB folder:', DATA_DIR)
print('Keep this Colab runtime active while using JASSIM.')
